# 04 — Probability Calibration

**Rift: Graph ML for Fraud Detection, Replay, and Audit**

This notebook demonstrates why calibration matters and compares raw, Platt-scaled, and isotonic-calibrated scores on ECE and Brier metrics.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AngelP17/Rift/blob/main/notebooks/04_calibration.ipynb)


## Environment Setup


In [ ]:
# Clone repo and install dependencies (run once)
import subprocess, sys, os

REPO = "https://github.com/AngelP17/Rift.git"
if not os.path.exists("/content/Rift"):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/Rift"], check=True)

os.chdir("/content/Rift")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "polars", "numpy", "pandas", "scikit-learn", "xgboost", "duckdb",
    "shap", "structlog", "python-dotenv", "rich", "jinja2", "pyarrow"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "torch", "--index-url", "https://download.pytorch.org/whl/cpu"], check=False)
sys.path.insert(0, "src")

try:
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    device = "cpu"
print(f"Device: {device}")


## Train a Model and Get Raw Scores


In [ ]:
import numpy as np
from data.generator import generate_transactions
from features.engine import build_features, get_feature_matrix, FEATURE_COLUMNS
from data.splits import split_data
from models.baseline_xgb import TabularXGBoost

df = generate_transactions(n_txns=20_000, fraud_rate=0.03, seed=42)
df_feat = build_features(df)
splits = split_data(df_feat, strategy="temporal")

X_train = get_feature_matrix(splits["train"]).to_numpy().astype(np.float32)
y_train = splits["train"]["is_fraud"].to_numpy().astype(np.float32)
X_val = get_feature_matrix(splits["val"]).to_numpy().astype(np.float32)
y_val = splits["val"]["is_fraud"].to_numpy().astype(np.float32)
X_test = get_feature_matrix(splits["test"]).to_numpy().astype(np.float32)
y_test = splits["test"]["is_fraud"].to_numpy().astype(np.float32)

model = TabularXGBoost(num_rounds=200)
model.fit(X_train, y_train, X_val, y_val)
raw_scores = model.predict_proba(X_test)
print(f"Raw score range: [{raw_scores.min():.4f}, {raw_scores.max():.4f}]")


## Apply Calibration Methods


In [ ]:
from models.calibrate import PlattCalibrator, IsotonicCalibrator
from models.metrics import expected_calibration_error, compute_all_metrics

val_scores = model.predict_proba(X_val)

platt = PlattCalibrator().fit(val_scores, y_val)
isotonic = IsotonicCalibrator().fit(val_scores, y_val)

platt_scores = platt.calibrate(raw_scores)
isotonic_scores = isotonic.calibrate(raw_scores)

print("=== Calibration Comparison ===")
for name, scores in [("Raw", raw_scores), ("Platt", platt_scores), ("Isotonic", isotonic_scores)]:
    ece = expected_calibration_error(y_test, scores)
    from sklearn.metrics import brier_score_loss
    brier = brier_score_loss(y_test, scores)
    print(f"  {name:10s} | ECE: {ece:.4f} | Brier: {brier:.4f}")


## Reliability Curves

A reliability curve shows predicted probability vs actual fraction of positives. A well-calibrated model follows the diagonal.


In [ ]:
from models.metrics import reliability_curve

for name, scores in [("Raw", raw_scores), ("Platt", platt_scores), ("Isotonic", isotonic_scores)]:
    frac_pos, mean_pred = reliability_curve(y_test, scores, n_bins=10)
    print(f"\n{name} Reliability Curve:")
    print(f"  Mean predicted: {mean_pred}")
    print(f"  Fraction positive: {frac_pos}")


## Impact on Decision Quality

Calibration does not change ranking (PR-AUC stays the same) but improves the operational meaning of scores.


In [ ]:
from sklearn.metrics import average_precision_score

for name, scores in [("Raw", raw_scores), ("Platt", platt_scores), ("Isotonic", isotonic_scores)]:
    pr_auc = average_precision_score(y_test, scores)
    ece = expected_calibration_error(y_test, scores)
    print(f"{name:10s} | PR-AUC: {pr_auc:.4f} | ECE: {ece:.4f}")
    
print("\nKey insight: PR-AUC is preserved while ECE improves significantly.")
